In [18]:
pip install ucimlrepo


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [19]:
from ucimlrepo import fetch_ucirepo
import numpy as np
import pandas as pd

# fetch dataset
wholesale_customers = fetch_ucirepo(id=292)

# features
X = wholesale_customers.data.features.copy()

# Drop categorical columns
X = X.drop(columns=["Channel"])

# Convert to numpy
X = X.values

# Normalize
X = (X - X.mean(axis=0)) / X.std(axis=0)

print("Shape:", X.shape)

Shape: (440, 6)


# K-Means from Scratch

In [20]:
class KMeans:
    def __init__(self, k=3, max_iters=100):
        self.k = k
        self.max_iters = max_iters

    def initialize_centroids(self, X):
        indices = np.random.choice(len(X), self.k, replace=False)
        return X[indices]

    def assign_clusters(self, X, centroids):
        distances = np.linalg.norm(X[:, np.newaxis] - centroids, axis=2)
        return np.argmin(distances, axis=1)

    def update_centroids(self, X, labels):
        centroids = []
        for i in range(self.k):
            cluster_points = X[labels == i]

            # Handle empty cluster
            if len(cluster_points) == 0:
                centroids.append(X[np.random.randint(0, len(X))])
            else:
                centroids.append(cluster_points.mean(axis=0))

        return np.array(centroids)

    def fit(self, X):
        self.centroids = self.initialize_centroids(X)

        for _ in range(self.max_iters):
            self.labels = self.assign_clusters(X, self.centroids)
            new_centroids = self.update_centroids(X, self.labels)

            if np.allclose(self.centroids, new_centroids):
                break

            self.centroids = new_centroids

        return self.labels, self.centroids

In [21]:
# run K-Means
k = 3

kmeans = KMeans(k=k)
labels, centroids = kmeans.fit(X)

print("K-Means centroids:\n", centroids)

K-Means centroids:
 [[ 0.10431338  2.83389626  2.89300865  0.11737131  2.84548307  1.29774939]
 [ 1.45256232 -0.16969649 -0.28301102  0.95868327 -0.42076661  0.26930694]
 [-0.34794049 -0.16204002 -0.13968646 -0.23309463 -0.104009   -0.15555262]]


# GMM from Scratch